# Movie Overview Embeddings

## Cilj projekta

Cilj projekta je poređenje tri Transformer embedding modela za generisanje jednog semantičkog embeddinga po filmu, korišćenjem njegovog tekstualnog opisa iz kolone "overview".


### Modeli

1. sentence-transformers/all-MiniLM-L6-v2
2. Alibaba-NLP/gte-modernbert-base
3. sentence-transformers/all-mpnet-base-v2

Primarni ulaz u modele biće isključivo tekst iz kolone overview. Naslov, jezik, popularnost, ocene i ostali metapodaci neće biti deo embedding teksta.

In [1]:
import re
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

In [2]:
random_state = 42

project_root = Path.cwd()
dataset_path = project_root / "data" / "raw" / "top_rated_movies.csv"

model_ids = {
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",
    "gte_modernbert": "Alibaba-NLP/gte-modernbert-base",
    "mpnet": "sentence-transformers/all-mpnet-base-v2",
}

In [3]:
df = pd.read_csv(dataset_path)


In [4]:
display(df.head())

print("Dimenzije dataseta:")
print(df.shape)

print("\nKolone:")
print(df.columns.tolist())

,adult,id,original_language,original_title,overview,popularity,release_date,title,vote_average,vote_count
0,False,278,en,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,46.3708,1994-09-23,The Shawshank Redemption,8.718,30171
1,False,238,en,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",42.0006,1972-03-14,The Godfather,8.686,22787
2,False,240,en,The Godfather Part II,In the continuing saga of the Corleone crime f...,26.8671,1974-12-20,The Godfather Part II,8.571,13812
3,False,424,en,Schindler's List,The true story of how businessman Oskar Schind...,24.2944,1993-12-15,Schindler's List,8.567,17341
4,False,389,en,12 Angry Men,The defense and the prosecution have rested an...,19.4971,1957-04-10,12 Angry Men,8.559,9908


Dimenzije dataseta:
(10000, 10)

Kolone:
['adult', 'id', 'original_language', 'original_title', 'overview', 'popularity', 'release_date', 'title', 'vote_average', 'vote_count']


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   adult              10000 non-null  bool   
 1   id                 10000 non-null  int64  
 2   original_language  10000 non-null  str    
 3   original_title     10000 non-null  str    
 4   overview           9998 non-null   str    
 5   popularity         10000 non-null  float64
 6   release_date       9997 non-null   str    
 7   title              10000 non-null  str    
 8   vote_average       10000 non-null  float64
 9   vote_count         10000 non-null  int64  
dtypes: bool(1), float64(2), int64(2), str(5)
memory usage: 713.0 KB


In [6]:
missing_values = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame(name="missing_count")
)

missing_values["missing_percentage"] = (
    missing_values["missing_count"] / len(df) * 100
).round(2)

display(missing_values)

,missing_count,missing_percentage
release_date,3,0.03
overview,2,0.02
id,0,0.00
adult,0,0.00
original_title,0,0.00
original_language,0,0.00
popularity,0,0.00
title,0,0.00
vote_average,0,0.00
vote_count,0,0.00


In [7]:
df["movie_id"] = df["id"].astype("int64")

required_columns = {
    "movie_id",
    "title",
    "overview",
}

missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise ValueError(
        f"Nedostaju obavezne kolone: {sorted(missing_columns)}"
    )

print(f"Broj filmova: {len(df)}")
print(f"Jedinstveni movie_id: {df['movie_id'].nunique()}")
print(f"Nedostajući naslovi: {df['title'].isna().sum()}")
print(f"Nedostajući opisi: {df['overview'].isna().sum()}")

display(
    df[
        [
            "movie_id",
            "title",
            "overview",
            "original_language",
            "vote_average",
        ]
    ].head()
)



Broj filmova: 10000
Jedinstveni movie_id: 9969
Nedostajući naslovi: 0
Nedostajući opisi: 2


,movie_id,title,overview,original_language,vote_average
0,278,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,en,8.718
1,238,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",en,8.686
2,240,The Godfather Part II,In the continuing saga of the Corleone crime f...,en,8.571
3,424,Schindler's List,The true story of how businessman Oskar Schind...,en,8.567
4,389,12 Angry Men,The defense and the prosecution have rested an...,en,8.559


In [8]:
rows_before = len(df)

df = (
    df.drop_duplicates(subset="movie_id", keep="first")
    .reset_index(drop=True)
)

rows_removed = rows_before - len(df)

assert df["movie_id"].notna().all()
assert df["movie_id"].is_unique

print(f"Uklonjeno dupliranih filmova: {rows_removed}")
print(f"Preostalo filmova: {len(df)}")
print("Gotov proces")

Uklonjeno dupliranih filmova: 31
Preostalo filmova: 9969
Gotov proces


In [9]:
rows_before = len(df)

df = df[
    df["overview"].notna()
    & df["overview"].astype(str).str.strip().ne("")
].copy()

df = df.reset_index(drop=True)


In [10]:
print(f"Broj filmova: {len(df)}")
print(f"Jedinstveni movie_id: {df['movie_id'].nunique()}")
print(f"Nedostajući naslovi: {df['title'].isna().sum()}")
print(f"Nedostajući opisi: {df['overview'].isna().sum()}")

display(
    df[
        [
            "movie_id",
            "title",
            "overview",
            "original_language",
            "vote_average",
        ]
    ].head()
    
)

Broj filmova: 9967
Jedinstveni movie_id: 9967
Nedostajući naslovi: 0
Nedostajući opisi: 0


,movie_id,title,overview,original_language,vote_average
0,278,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,en,8.718
1,238,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...",en,8.686
2,240,The Godfather Part II,In the continuing saga of the Corleone crime f...,en,8.571
3,424,Schindler's List,The true story of how businessman Oskar Schind...,en,8.567
4,389,12 Angry Men,The defense and the prosecution have rested an...,en,8.559


In [11]:
def clean_overview(text: str) -> str:
    text = BeautifulSoup(str(text), "html.parser").get_text(" ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [12]:
df["embedding_text"] = df["overview"].apply(clean_overview)

df = df[df["embedding_text"].ne("")].copy()
df = df.reset_index(drop=True)

In [13]:
display(
    df[["overview", "embedding_text"]].sample(
        n=min(10, len(df)),
        random_state=4,
    )
)

,overview,embedding_text
4171,The great King Gurumes is searching for the Dr...,The great King Gurumes is searching for the Dr...
792,A private detective takes on a case that invol...,A private detective takes on a case that invol...
373,"While waiting for a kidney transplant, a young...","While waiting for a kidney transplant, a young..."
9392,Astronaut Scorch Supernova finds himself caugh...,Astronaut Scorch Supernova finds himself caugh...
4214,A group of 12 teenagers from various backgroun...,A group of 12 teenagers from various backgroun...
4917,Strange things begin to occur as a tiny Califo...,Strange things begin to occur as a tiny Califo...
9260,A young girl sent to live with her father and ...,A young girl sent to live with her father and ...
3014,"Romulus and Remus, two shepherds and loyal bro...","Romulus and Remus, two shepherds and loyal bro..."
5255,Teenager Winnie Foster is growing up in a smal...,Teenager Winnie Foster is growing up in a smal...
6985,The patriarch of a wealthy and powerful family...,The patriarch of a wealthy and powerful family...


In [14]:
output_path = Path("data/processed/movies_cleaned.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

print(df.shape)
print(f"Sačuvano u: {output_path}")

(9967, 12)
Sačuvano u: data\processed\movies_cleaned.csv
